# Project 3: Recommender System with Matrix Factorization

## Introduction

Build a Netflix-style recommendation system using **Matrix Factorization** - the technique that won the Netflix Prize!

### The Problem

Given a sparse user-item rating matrix $R$ (e.g., users × movies), predict missing ratings.

### Matrix Factorization Approach

Factor $R \approx P \times Q^T$ where:
- $P$: user features (n_users × k latent factors)
- $Q$: item features (n_items × k latent factors)
- Predicted rating: $\hat{r}_{ui} = p_u \cdot q_i$

### Optimization Problem

Minimize:
$$\min_{P,Q} \sum_{(u,i) \in \text{observed}} (r_{ui} - p_u \cdot q_i)^2 + \lambda(\|P\|^2 + \|Q\|^2)$$

We'll use **Alternating Least Squares (ALS)** and **Stochastic Gradient Descent (SGD)**.

---

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_squared_error

np.random.seed(42)
plt.rcParams['figure.figsize'] = (12, 6)

## Part 1: Create Synthetic Movie Rating Data

In [ ]:
def generate_rating_data(n_users=100, n_items=50, n_factors=5, sparsity=0.7):
    """Generate synthetic user-item rating matrix."""
    # True latent factors
    P_true = np.random.rand(n_users, n_factors) * 2
    Q_true = np.random.rand(n_items, n_factors) * 2
    
    # Generate ratings
    R_full = P_true @ Q_true.T
    R_full = np.clip(R_full, 1, 5)  # Clip to 1-5 star range
    
    # Create sparsity (missing ratings)
    mask = np.random.rand(n_users, n_items) > sparsity
    R = R_full.copy()
    R[~mask] = 0  # 0 = missing rating
    
    return R, R_full, mask

# Generate data
R, R_full, mask = generate_rating_data(n_users=100, n_items=50, n_factors=5, sparsity=0.7)

print(f"Rating matrix shape: {R.shape}")
print(f"Total possible ratings: {R.size}")
print(f"Observed ratings: {np.sum(mask)} ({np.sum(mask)/R.size*100:.1f}%)")
print(f"Sparsity: {(1 - np.sum(mask)/R.size)*100:.1f}%")

# Visualize
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

im1 = ax1.imshow(R_full, cmap='YlOrRd', aspect='auto')
ax1.set_title('True Complete Ratings', fontsize=13, fontweight='bold')
ax1.set_xlabel('Items (Movies)')
ax1.set_ylabel('Users')
plt.colorbar(im1, ax=ax1, label='Rating')

R_vis = R.copy()
R_vis[~mask] = np.nan
im2 = ax2.imshow(R_vis, cmap='YlOrRd', aspect='auto')
ax2.set_title('Observed Ratings (Sparse)', fontsize=13, fontweight='bold')
ax2.set_xlabel('Items (Movies)')
ax2.set_ylabel('Users')
plt.colorbar(im2, ax=ax2, label='Rating')

plt.tight_layout()
plt.show()

## Part 2: Matrix Factorization with ALS

In [ ]:
class MatrixFactorizationALS:
    """
    Matrix Factorization using Alternating Least Squares (ALS).
    """
    
    def __init__(self, n_factors=10, n_iterations=20, reg_lambda=0.1):
        self.n_factors = n_factors
        self.n_iterations = n_iterations
        self.reg_lambda = reg_lambda
        self.P = None
        self.Q = None
        
    def fit(self, R, mask):
        """
        Fit model using ALS.
        
        Args:
            R: Rating matrix (n_users × n_items)
            mask: Boolean mask of observed ratings
        """
        n_users, n_items = R.shape
        
        # Initialize randomly
        self.P = np.random.rand(n_users, self.n_factors)
        self.Q = np.random.rand(n_items, self.n_factors)
        
        train_errors = []
        
        for iteration in range(self.n_iterations):
            # Fix Q, update P
            for u in range(n_users):
                # Items rated by user u
                items_u = mask[u, :]
                if np.sum(items_u) > 0:
                    Q_u = self.Q[items_u, :]
                    r_u = R[u, items_u]
                    
                    # Solve: (Q^T Q + λI) p_u = Q^T r_u
                    A = Q_u.T @ Q_u + self.reg_lambda * np.eye(self.n_factors)
                    b = Q_u.T @ r_u
                    self.P[u, :] = np.linalg.solve(A, b)
            
            # Fix P, update Q
            for i in range(n_items):
                # Users who rated item i
                users_i = mask[:, i]
                if np.sum(users_i) > 0:
                    P_i = self.P[users_i, :]
                    r_i = R[users_i, i]
                    
                    # Solve: (P^T P + λI) q_i = P^T r_i
                    A = P_i.T @ P_i + self.reg_lambda * np.eye(self.n_factors)
                    b = P_i.T @ r_i
                    self.Q[i, :] = np.linalg.solve(A, b)
            
            # Compute training error
            R_pred = self.P @ self.Q.T
            error = np.sqrt(np.sum(mask * (R - R_pred)**2) / np.sum(mask))
            train_errors.append(error)
            
            if (iteration + 1) % 5 == 0:
                print(f"Iteration {iteration+1}/{self.n_iterations}, RMSE: {error:.4f}")
        
        return train_errors
    
    def predict(self):
        """Predict all ratings."""
        return self.P @ self.Q.T

# Fit model
mf_als = MatrixFactorizationALS(n_factors=5, n_iterations=20, reg_lambda=0.1)
errors = mf_als.fit(R, mask)

# Predict
R_pred_als = mf_als.predict()
R_pred_als = np.clip(R_pred_als, 1, 5)

print(f"\nFinal RMSE on observed ratings: {errors[-1]:.4f}")

## Part 3: Visualize Results

In [ ]:
# Plot training curve
plt.figure(figsize=(10, 5))
plt.plot(range(1, len(errors)+1), errors, 'o-', linewidth=2, markersize=6)
plt.xlabel('Iteration', fontsize=12)
plt.ylabel('RMSE', fontsize=12)
plt.title('ALS Training Convergence', fontsize=14, fontweight='bold')
plt.grid(alpha=0.3)
plt.tight_layout()
plt.show()

# Compare predictions
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

im0 = axes[0].imshow(R_full, cmap='YlOrRd', aspect='auto', vmin=1, vmax=5)
axes[0].set_title('Ground Truth', fontsize=13, fontweight='bold')
axes[0].set_ylabel('Users')
axes[0].set_xlabel('Items')
plt.colorbar(im0, ax=axes[0])

im1 = axes[1].imshow(R_pred_als, cmap='YlOrRd', aspect='auto', vmin=1, vmax=5)
axes[1].set_title('ALS Predictions', fontsize=13, fontweight='bold')
axes[1].set_xlabel('Items')
plt.colorbar(im1, ax=axes[1])

error_map = np.abs(R_full - R_pred_als)
im2 = axes[2].imshow(error_map, cmap='Reds', aspect='auto')
axes[2].set_title('Absolute Error', fontsize=13, fontweight='bold')
axes[2].set_xlabel('Items')
plt.colorbar(im2, ax=axes[2])

plt.tight_layout()
plt.show()

print(f"Mean Absolute Error: {np.mean(error_map):.4f}")
print(f"RMSE on all entries: {np.sqrt(np.mean((R_full - R_pred_als)**2)):.4f}")

## Part 4: Matrix Factorization with SGD

In [ ]:
class MatrixFactorizationSGD:
    """
    Matrix Factorization using Stochastic Gradient Descent.
    """
    
    def __init__(self, n_factors=10, learning_rate=0.01, n_epochs=50, reg_lambda=0.02):
        self.n_factors = n_factors
        self.learning_rate = learning_rate
        self.n_epochs = n_epochs
        self.reg_lambda = reg_lambda
        self.P = None
        self.Q = None
        
    def fit(self, R, mask):
        n_users, n_items = R.shape
        
        # Initialize
        self.P = np.random.rand(n_users, self.n_factors) * 0.1
        self.Q = np.random.rand(n_items, self.n_factors) * 0.1
        
        # Get observed ratings
        observed = np.argwhere(mask)
        
        train_errors = []
        
        for epoch in range(self.n_epochs):
            # Shuffle
            np.random.shuffle(observed)
            
            # SGD updates
            for u, i in observed:
                # Prediction
                r_pred = self.P[u, :] @ self.Q[i, :]
                error = R[u, i] - r_pred
                
                # Gradients
                grad_P = -2 * error * self.Q[i, :] + 2 * self.reg_lambda * self.P[u, :]
                grad_Q = -2 * error * self.P[u, :] + 2 * self.reg_lambda * self.Q[i, :]
                
                # Updates
                self.P[u, :] -= self.learning_rate * grad_P
                self.Q[i, :] -= self.learning_rate * grad_Q
            
            # Compute RMSE
            R_pred = self.P @ self.Q.T
            rmse = np.sqrt(np.sum(mask * (R - R_pred)**2) / np.sum(mask))
            train_errors.append(rmse)
            
            if (epoch + 1) % 10 == 0:
                print(f"Epoch {epoch+1}/{self.n_epochs}, RMSE: {rmse:.4f}")
        
        return train_errors
    
    def predict(self):
        return self.P @ self.Q.T

# Fit SGD model
mf_sgd = MatrixFactorizationSGD(n_factors=5, learning_rate=0.01, n_epochs=50, reg_lambda=0.02)
errors_sgd = mf_sgd.fit(R, mask)

R_pred_sgd = mf_sgd.predict()
R_pred_sgd = np.clip(R_pred_sgd, 1, 5)

print(f"\nFinal RMSE: {errors_sgd[-1]:.4f}")

## Part 5: Compare ALS vs SGD

In [ ]:
# Plot comparison
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(16, 5))

# Training curves
ax1.plot(range(1, len(errors)+1), errors, 'o-', label='ALS', linewidth=2, markersize=6)
ax1.plot(range(1, len(errors_sgd)+1), errors_sgd, 's-', label='SGD', linewidth=2, markersize=5)
ax1.set_xlabel('Iteration/Epoch', fontsize=12)
ax1.set_ylabel('RMSE', fontsize=12)
ax1.set_title('Training Convergence: ALS vs SGD', fontsize=14, fontweight='bold')
ax1.legend(fontsize=11)
ax1.grid(alpha=0.3)

# Prediction comparison
rmse_als = np.sqrt(np.mean((R_full - R_pred_als)**2))
rmse_sgd = np.sqrt(np.mean((R_full - R_pred_sgd)**2))
mae_als = np.mean(np.abs(R_full - R_pred_als))
mae_sgd = np.mean(np.abs(R_full - R_pred_sgd))

methods = ['ALS', 'SGD']
rmse_vals = [rmse_als, rmse_sgd]
mae_vals = [mae_als, mae_sgd]

x = np.arange(len(methods))
width = 0.35

ax2.bar(x - width/2, rmse_vals, width, label='RMSE', alpha=0.8)
ax2.bar(x + width/2, mae_vals, width, label='MAE', alpha=0.8)
ax2.set_ylabel('Error', fontsize=12)
ax2.set_title('Final Prediction Errors', fontsize=14, fontweight='bold')
ax2.set_xticks(x)
ax2.set_xticklabels(methods)
ax2.legend()
ax2.grid(axis='y', alpha=0.3)

plt.tight_layout()
plt.show()

print("\n=== Performance Summary ===")
print(f"ALS  - RMSE: {rmse_als:.4f}, MAE: {mae_als:.4f}")
print(f"SGD  - RMSE: {rmse_sgd:.4f}, MAE: {mae_sgd:.4f}")

## Part 6: Making Recommendations

In [ ]:
def recommend_items(user_id, R_pred, R_observed, mask, top_k=5):
    """
    Recommend top-k items for a user.
    """
    # Get predictions for user
    user_predictions = R_pred[user_id, :]
    
    # Exclude already rated items
    unrated_items = ~mask[user_id, :]
    
    # Get top-k unrated items
    item_scores = user_predictions * unrated_items
    top_items = np.argsort(item_scores)[::-1][:top_k]
    
    return top_items, user_predictions[top_items]

# Example: recommend for user 0
user_id = 0
top_items, scores = recommend_items(user_id, R_pred_als, R, mask, top_k=10)

print(f"\n=== Top 10 Recommendations for User {user_id} ===")
print(f"{'Item ID':<10} {'Predicted Rating':<20} {'True Rating':<15}")
print("-" * 45)
for item, score in zip(top_items, scores):
    true_rating = R_full[user_id, item]
    print(f"{item:<10} {score:<20.2f} {true_rating:<15.2f}")

# Visualize user's profile
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 4))

# Observed vs predicted ratings
rated_items = np.where(mask[user_id, :])[0]
observed_ratings = R[user_id, rated_items]
predicted_ratings = R_pred_als[user_id, rated_items]

ax1.scatter(observed_ratings, predicted_ratings, alpha=0.6, s=50)
ax1.plot([1, 5], [1, 5], 'r--', label='Perfect prediction')
ax1.set_xlabel('Observed Rating', fontsize=11)
ax1.set_ylabel('Predicted Rating', fontsize=11)
ax1.set_title(f'User {user_id}: Observed vs Predicted', fontsize=12, fontweight='bold')
ax1.legend()
ax1.grid(alpha=0.3)

# All predictions
all_predictions = R_pred_als[user_id, :]
ax2.bar(range(len(all_predictions)), all_predictions, alpha=0.6, color='steelblue')
ax2.scatter(rated_items, all_predictions[rated_items], color='red', s=50, 
           label='Observed', zorder=5)
ax2.scatter(top_items, all_predictions[top_items], color='green', s=50, 
           label='Recommended', zorder=5, marker='^')
ax2.set_xlabel('Item ID', fontsize=11)
ax2.set_ylabel('Predicted Rating', fontsize=11)
ax2.set_title(f'User {user_id}: All Predicted Ratings', fontsize=12, fontweight='bold')
ax2.legend()
ax2.grid(axis='y', alpha=0.3)

plt.tight_layout()
plt.show()

## Part 7: Latent Factor Interpretation

In [ ]:
# Visualize latent factors
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

# User factors
im1 = ax1.imshow(mf_als.P[:20, :], cmap='RdBu_r', aspect='auto')
ax1.set_title('User Latent Factors (First 20 Users)', fontsize=12, fontweight='bold')
ax1.set_xlabel('Latent Factor')
ax1.set_ylabel('User ID')
plt.colorbar(im1, ax=ax1)

# Item factors
im2 = ax2.imshow(mf_als.Q.T, cmap='RdBu_r', aspect='auto')
ax2.set_title('Item Latent Factors', fontsize=12, fontweight='bold')
ax2.set_xlabel('Item ID')
ax2.set_ylabel('Latent Factor')
plt.colorbar(im2, ax=ax2)

plt.tight_layout()
plt.show()

print("\nLatent factors capture user preferences and item characteristics.")
print("For movies: factors might represent genre, director style, etc.")
print("For users: factors represent preferences along those dimensions.")

## Summary

### What We Learned

1. **Matrix Factorization**: $R \approx PQ^T$ factorizes rating matrix into user/item features
2. **ALS**: Alternates between fixing P and Q, solving least-squares
3. **SGD**: Updates factors via gradient descent on observed ratings
4. **Regularization**: $\lambda$ prevents overfitting
5. **Cold Start**: New users/items have no ratings (need content-based features)

### Real-World Applications

- **Netflix**: Movie recommendations
- **Amazon**: Product recommendations
- **Spotify**: Music recommendations
- **LinkedIn**: Job/connection recommendations

### When to Use Matrix Factorization

✅ **Good for:**
- Sparse rating matrices
- Scalable to millions of users/items
- Capturing latent preferences

❌ **Limitations:**
- Cold start problem
- Doesn't use content features
- Linear interactions only

### Modern Extensions

- **Neural Collaborative Filtering**: Deep learning for non-linear interactions
- **Factorization Machines**: Incorporate side information
- **Temporal Models**: Account for time-varying preferences

### Next Steps

- **Project 4: LASSO** - Learn sparse optimization for feature selection
- **Project 5: Logistic Regression** - Classification with optimization

---

**Great job!** You've built a recommendation system like the ones powering Netflix and Amazon! 🎬🛒